# TSCGlue Regression Demo

Trains `TSCGlueRegressor` on a univariate TSER dataset (FloodModeling1: 471 train / 202 test, 1 channel, length 266).

In [1]:
import numpy as np
import pandas as pd
from aeon.datasets import load_regression
from sklearn.metrics import mean_squared_error, r2_score

from tscglue.models import TSCGlueRegressor

In [2]:
DATASET = "FloodModeling1"

In [3]:
X_train, y_train = load_regression(DATASET, split="train")
X_test, y_test = load_regression(DATASET, split="test")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: [{y_train.min():.4f}, {y_train.max():.4f}]")

Train: (471, 1, 266), Test: (202, 1, 266)
Target range: [0.4050, 0.6750]


In [4]:
reg = TSCGlueRegressor(n_jobs=4, verbose=1)
reg.fit(X_train, y_train)

[1.59s] Fit+transformed multirocket_s_1756208465 features (471, 49728) (178.69 MB) dtype=float64 in 1.5876s
[6.67s] Fit+transformed hydra_s_520560508 features (471, 6144) (22.08 MB) dtype=float64 in 5.0845s
[10.04s] Fit+transformed quant features (471, 2481) (8.92 MB) dtype=float64 in 3.3670s
[20.02s] Fit+transformed rdst_s_788558511 features (471, 30000) (107.80 MB) dtype=float64 in 9.9766s
[25.74s] Fit+transformed rstsf-random_s_1954798752 features (471, 13027) (46.81 MB) dtype=float64 in 5.7213s
[32.11s] Fit+transformed mantis features (471, 512) (1.84 MB) dtype=float64 in 6.3647s
[48.53s] Fit+transformed chronos2 features (471, 1536) (5.52 MB) dtype=float64 in 16.4258s
[85.55s] Fit+transformed tsfresh features (471, 777) (2.79 MB) dtype=float64 in 37.0143s
[172.08s] Fit+transformed drcif_s_1979705049 features (471, 12180) (43.77 MB) dtype=float64 in 86.5359s
[209.52s] OOF  multirockethydra-etr_s_464797957                RMSE  0.0110   MAE  0.0046   R²     0.7453   robust R²   0.745

TSCGlueRegressor(n_jobs=4, verbose=1)

In [5]:
y_pred = reg.predict(X_test)
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.4f}")

[0.64s] Computed multirocket_s_1756208465 features (202, 49728) in 0.6221s
[4.34s] Computed hydra_s_520560508 features (202, 6144) in 3.7065s
[7.75s] Computed quant features (202, 2481) in 3.4053s
[11.27s] Computed rdst_s_788558511 features (202, 30000) in 3.5202s
[13.46s] Computed rstsf-random_s_1954798752 features (202, 13027) in 2.1934s
[19.04s] Computed mantis features (202, 512) in 5.5729s
[29.98s] Computed chronos2 features (202, 1536) in 10.9421s
[48.00s] Computed tsfresh features (202, 777) in 18.0249s
[103.59s] Computed drcif_s_1979705049 features (202, 12180) in 55.5856s
Test RMSE: 0.0039
Test R²:   0.9574


In [6]:
# Per-model out-of-fold scores (base models + stacker)
pd.DataFrame(reg.summary())

,model,level,oof_rmse,oof_mae,oof_r2,oof_r2_robust,train_time
0,multirockethydra-etr_s_464797957,0,1.104861e-02,0.004633,7.452559e-01,0.745256,"[8.051381197998126, 8.110059820002789, 8.17111..."
1,multirockethydra-ridgecv_s_1204601876,0,1.561046e-02,0.004385,4.914659e-01,0.931405,"[7.769858432002366, 8.129039307001221, 5.14842..."
2,multirockethydra-clipped-ridgecv_s_1892542538,0,8.300633e-03,0.003945,8.562158e-01,0.856216,"[16.125142340999446, 15.175959964999493, 15.73..."
3,quant-etr_s_2126260716,0,1.121923e-02,0.004741,7.373274e-01,0.737327,"[10.346371272997203, 6.195378671000071, 4.4197..."
4,quant-ridgecv_s_1841892493,0,6.177654e-03,0.003988,9.203591e-01,0.936698,"[7.154849119000573, 8.047564317999786, 8.31662..."
5,quant-clipped-ridgecv_s_44536905,0,8.357070e-03,0.004170,8.542539e-01,0.854254,"[11.405378706000192, 9.187044816000707, 9.3871..."
6,rdst-etr_s_535350907,0,1.899505e-02,0.012047,2.470440e-01,0.247044,"[7.26631947200076, 6.395503572999587, 5.801495..."
7,rdst-ridgecv_s_830972893,0,7.846592e+06,361551.869445,-1.284846e+17,0.434184,"[4.356275766000181, 5.0375597450001806, 4.4826..."
8,rdst-clipped-ridgecv_s_833455768,0,1.609567e-02,0.009371,4.593617e-01,0.459362,"[14.083910556000774, 12.986695786999917, 14.33..."
9,rstsf-random-etr_s_122261377,0,1.008039e-02,0.004320,7.879475e-01,0.787947,"[3.4076813320025394, 3.763440951002849, 3.0626..."


In [7]:
reg.cleanup()